# Домашняя работа 5. Преобразование и подготовка признаков

Датасет — объявления Airbnb в Нью-Йорке (фиксированная версия `listings.csv`). Работа начинается с загрузки исходного файла и не опирается на результаты HW4.

План: рабочая таблица → векторные и строковые преобразования → `map` → `loc` → `apply` и `lambda` → One-Hot Encoding → масштабирование → финальная таблица признаков.

In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler

pd.set_option("display.max_columns", 40)

## 1. Подготовка рабочей таблицы

Загружаем исходный файл в `df` и создаём отдельную копию `df_work` с нужными столбцами. `.copy()` гарантирует, что изменения `df_work` не затронут `df`.

In [2]:
df = pd.read_csv("listings.csv")

work_columns = [
    "id",
    "name",
    "neighbourhood_group",
    "room_type",
    "price",
    "minimum_nights",
    "number_of_reviews",
    "reviews_per_month",
    "availability_365",
]

df_work = df[work_columns].copy()

print("Размер df:", df.shape)
print("Размер df_work:", df_work.shape)

Размер df: (30555, 19)
Размер df_work: (30555, 9)


In [3]:
df_work.dtypes

id                       int64
name                       str
neighbourhood_group        str
room_type                  str
price                  float64
minimum_nights         float64
number_of_reviews        int64
reviews_per_month      float64
availability_365         int64
dtype: object

In [4]:
print("Пропуски в выбранных признаках:")
print(df_work.isna().sum())

Пропуски в выбранных признаках:
id                        0
name                      1
neighbourhood_group       0
room_type                 0
price                  8758
minimum_nights            2
number_of_reviews         0
reviews_per_month      8616
availability_365          0
dtype: int64


In [5]:
rows_before = len(df_work)
df_work = df_work.dropna(subset=["price", "minimum_nights"]).copy()

print("Строк до удаления:", rows_before)
print("Строк после удаления:", len(df_work))
print("Удалено строк:", rows_before - len(df_work))
print()
print("Пропуски после удаления:")
print(df_work.isna().sum())
print()
print("Размер исходного df не изменился:", df.shape)

Строк до удаления: 30555
Строк после удаления: 21796
Удалено строк: 8759

Пропуски после удаления:
id                        0
name                      0
neighbourhood_group       0
room_type                 0
price                     0
minimum_nights            0
number_of_reviews         0
reviews_per_month      6133
availability_365          0
dtype: int64

Размер исходного df не изменился: (30555, 19)


**Вывод.** В рабочую таблицу вошли 9 столбцов. Пропуски были в `price` (8 758), `reviews_per_month` (8 616), `minimum_nights` (2) и `name` (1). После удаления строк без цены или минимального срока осталось 21 796 объявлений; в этих строках нет и пропусков в `name`. Пропуски остались только в `reviews_per_month` — это объявления без отзывов, их учтём в заданиях 5 и 8. Исходный `df` не изменился.

## 2. Векторные и строковые преобразования

### 2.1. Стоимость недели

Умножаем весь столбец на 7 одной операцией — без цикла и `apply`.

In [6]:
df_work["price_per_week"] = df_work["price"] * 7

df_work[["price", "price_per_week"]].head()

,price,price_per_week
0,468.0,3276.0
1,188.0,1316.0
2,431.0,3017.0
3,257.0,1799.0
4,170.0,1190.0


### 2.2. Длина названия

Используем строковый аксессор `.str`: `.str.len()` считает длину каждой строки столбца. Если бы название отсутствовало, результат был бы пропуском, а не ошибкой.

In [7]:
df_work["name_length"] = df_work["name"].str.len()

df_work[["name", "name_length"]].head()

,name,name_length
0,"The Broome, Deluxe Queen",24
1,"M Social Hotel New York Downtown, Deluxe king ...",50
2,"ModernHaus SoHo, SoHo deluxe king",33
3,Superior 1-bedroom apartment w/ den & 2 queens,46
4,Chic boutique hotel stay | Ameritania Hotel,43


In [8]:
df_work["name_length"].describe().round(1)

count    21796.0
mean        37.1
std         10.9
min          2.0
25%         29.0
50%         39.0
75%         47.0
max        147.0
Name: name_length, dtype: float64

### 2.3. Вывод

`apply` здесь не нужен, потому что обе операции Pandas уже умеет выполнять над целым столбцом сразу: арифметика (`price * 7`) работает поэлементно для всего `Series`, а для строк есть готовые методы в `.str` (`.str.len()`). `apply` вызывает Python-функцию отдельно для каждой из 21 796 строк — это длиннее в записи и заметно медленнее. Векторная запись короче, быстрее и сразу понятна читателю, а `.str.len()` ещё и сам корректно обрабатывает пропуски. Поэтому векторный способ здесь проще и естественнее.

## 3. `map`, словарь и ручное перекодирование

### 3.1. Коды районов

`map` со словарём заменяет каждое значение столбца на соответствующее значение из словаря.

In [9]:
borough_codes = {
    "Manhattan": "MAN",
    "Brooklyn": "BRK",
    "Queens": "QNS",
    "Bronx": "BRX",
    "Staten Island": "SI",
}

df_work["borough_code"] = df_work["neighbourhood_group"].map(borough_codes)

df_work["borough_code"].value_counts()

borough_code
MAN    9771
BRK    7262
QNS    3638
BRX     829
SI      296
Name: count, dtype: int64

In [10]:
print("Пропусков в borough_code (значения, не найденные в словаре):", df_work["borough_code"].isna().sum())

Пропусков в borough_code (значения, не найденные в словаре): 0


Все 5 районов нашлись в словаре: пропусков нет, а количество объявлений по кодам совпадает с количеством по исходным названиям. Если бы в данных встретилось значение, которого нет в словаре, `map` вернул бы для него `NaN` — поэтому такая проверка полезна.

### 3.2. Бинарный признак `is_entire_home`

Словарь содержит только один тип жилья; все остальные типы после `map` получают `NaN`, который заменяем на 0.

In [11]:
df_work["is_entire_home"] = (
    df_work["room_type"]
    .map({"Entire home/apt": 1})
    .fillna(0)
    .astype(int)
)

pd.crosstab(df_work["room_type"], df_work["is_entire_home"])

is_entire_home,0,1
room_type,,
Entire home/apt,0,11537
Hotel room,480,0
Private room,9587,0
Shared room,192,0


Таблица частот подтверждает: единицу получили только объявления `Entire home/apt`, все остальные типы — ноль.

### 3.3. Почему не Label Encoding?

Кодирование `Manhattan = 0, Brooklyn = 1, Queens = 2, ...` создаёт между районами **порядок и расстояния, которых в данных нет**. Модель воспримет эти числа как обычную числовую шкалу: будет считать, что Queens «больше» Manhattan, что Brooklyn находится «посередине» между ними, а разница между Queens и Manhattan вдвое больше, чем между Brooklyn и Manhattan. Для линейных моделей и методов, основанных на расстояниях, это прямо искажает результат, а сама нумерация произвольна — при другом порядке кодов модель выучила бы другие зависимости.

Районы — **номинальный** признак без естественного порядка, поэтому для `neighbourhood_group` лучше подходит **One-Hot Encoding**: для каждого района создаётся отдельный столбец 0/1, и ни один район не оказывается «больше» другого.

## 4. `loc`: условное преобразование

Создаём `stay_type` по `minimum_nights`: каждой категории присваиваем значение через `loc` по логической маске.

In [12]:
df_work.loc[df_work["minimum_nights"] < 7, "stay_type"] = "short"
df_work.loc[
    (df_work["minimum_nights"] >= 7) & (df_work["minimum_nights"] <= 30),
    "stay_type",
] = "medium"
df_work.loc[df_work["minimum_nights"] > 30, "stay_type"] = "long"

print("Строк без категории:", df_work["stay_type"].isna().sum())
df_work["stay_type"].value_counts()

Строк без категории: 0


stay_type
medium    15988
short      5230
long        578
Name: count, dtype: int64

Проверяем по нескольку строк каждой категории, в том числе граничные значения 7 и 30.

In [13]:
df_work.groupby("stay_type").head(3)[["id", "minimum_nights", "stay_type"]].sort_values("stay_type")

,id,minimum_nights,stay_type
88,1294386769599107546,31.0,long
91,1065403714551533946,31.0,long
126,24784400,60.0,long
56,838263870384970566,30.0,medium
58,53117856,30.0,medium
59,1705099787293783835,30.0,medium
0,1279666081527148590,1.0,short
1,1543683559947083167,1.0,short
2,1559522395930572140,1.0,short


In [14]:
boundary_check = df_work[df_work["minimum_nights"].isin([6, 7, 30, 31])]
boundary_check.drop_duplicates("minimum_nights")[["minimum_nights", "stay_type"]].sort_values("minimum_nights")

,minimum_nights,stay_type
8261,6.0,short
2794,7.0,medium
56,30.0,medium
88,31.0,long


**Вывод.** Все объявления получили категорию. Больше всего `medium` (15 988), что ожидаемо: у большинства объявлений минимальный срок ровно 30 ночей. `short` — 5 230, `long` — 578. Граничные значения распределены правильно: 6 → `short`, 7 и 30 → `medium`, 31 → `long`.

**Почему `loc` удобнее цикла.** Каждое присваивание через `loc` обрабатывает сразу все подходящие строки одной операцией по маске: условие записано явно и читается как правило из задания. В цикле пришлось бы перебирать 21 796 строк по одной и менять таблицу построчно — это медленнее, длиннее и легко приводит к ошибкам (например, изменения в копии строки при `iterrows()` не попадают в таблицу).

## 5. `apply` и `lambda`

### 5.1. `review_activity` — функция по нескольким столбцам

Правило зависит сразу от двух столбцов, поэтому пишем отдельную функцию, которая получает целую строку, и применяем её через `apply(axis=1)`. Пропуск в `reviews_per_month` при ненулевом числе отзывов явно относим к `"rare"`.

In [15]:
def get_review_activity(row):
    """Определяет активность отзывов по числу отзывов и отзывам в месяц."""
    if row["number_of_reviews"] == 0:
        return "no_reviews"
    elif pd.notna(row["reviews_per_month"]) and row["reviews_per_month"] >= 2:
        return "active"
    else:
        # сюда попадают и reviews_per_month < 2, и пропуск при number_of_reviews > 0
        return "rare"


df_work["review_activity"] = df_work.apply(get_review_activity, axis=1)

df_work["review_activity"].value_counts()

review_activity
rare          12955
no_reviews     6133
active         2708
Name: count, dtype: int64

In [16]:
df_work.groupby("review_activity").head(2)[
    ["id", "number_of_reviews", "reviews_per_month", "review_activity"]
].sort_values("review_activity")

,id,number_of_reviews,reviews_per_month,review_activity
1,1543683559947083167,23,3.37,active
4,1394234603510215650,168,12.23,active
16,1043748763612320534,0,NaN,no_reviews
29,1645681291050081800,0,NaN,no_reviews
0,1279666081527148590,3,0.57,rare
2,1559522395930572140,3,0.54,rare


In [17]:
edge_case = df_work[(df_work["number_of_reviews"] > 0) & (df_work["reviews_per_month"].isna())]
print("Объявлений с отзывами, но без reviews_per_month:", len(edge_case))

Объявлений с отзывами, но без reviews_per_month: 0


**Вывод.** Больше всего объявлений с редкими отзывами (`rare`), на втором месте объявления без отзывов, а активных (2 и более отзывов в месяц) меньше всего. В этих данных нет объявлений с отзывами, но без `reviews_per_month`, однако функция всё равно корректно отнесла бы их к `"rare"` благодаря проверке `pd.notna(...)`: без неё сравнение `NaN >= 2` просто дало бы `False`, и логика оставалась бы неявной.

### 5.2. Признак через `lambda`: упоминание вида в названии

Создаём `mentions_view` — 1, если в названии объявления есть слово «view» (например, «City View», «River views!»), иначе 0. Вид из окна — частый аргумент для более высокой цены, поэтому признак может быть полезен.

In [18]:
df_work["mentions_view"] = df_work["name"].apply(
    lambda name: 1 if isinstance(name, str) and "view" in name.lower() else 0
)

print("Объявлений с упоминанием вида:", df_work["mentions_view"].sum())
df_work.loc[df_work["mentions_view"] == 1, ["name", "mentions_view"]].head()

Объявлений с упоминанием вида: 590


,name,mentions_view
14,"AMTD Idea Tribeca Hotel, 2 Doubles Tribeca view",1
36,Spacious suite with Times Square views,1
41,City view alcove suite one king bed,1
48,"Liberty View Brooklyn Hotel, Standard two double",1
53,"The Hoxton Williamsburg, Cozy Brooklyn view",1


**Почему здесь уместна `lambda`.** Правило очень короткое (одна строка), используется только один раз и не нужно больше нигде в ноутбуке — отдельная именованная функция через `def` только растянула бы код. Внутри `lambda` сразу видно всё правило, включая защиту от пустого названия (`isinstance(name, str)`).

Стоит отметить, что эту задачу можно решить и векторно: `df_work["name"].str.contains("view", case=False, na=False)`. Для больших данных векторный вариант предпочтительнее, а `lambda` уместна именно как небольшое разовое правило.

## 6. One-Hot Encoding для `room_type`

`pd.get_dummies` создаёт отдельный столбец 0/1 для каждого типа жилья. `dtype=int` делает значения числами 0/1 вместо `True`/`False`.

In [19]:
room_dummies = pd.get_dummies(df_work["room_type"], prefix="room", dtype=int)
room_dummies.columns = room_dummies.columns.str.replace(" ", "_").str.replace("/", "_")

df_work = pd.concat([df_work, room_dummies], axis=1)

df_work[["room_type"] + list(room_dummies.columns)].head()

,room_type,room_Entire_home_apt,room_Hotel_room,room_Private_room,room_Shared_room
0,Private room,0,0,1,0
1,Private room,0,0,1,0
2,Private room,0,0,1,0
3,Private room,0,0,1,0
4,Private room,0,0,1,0


In [20]:
print("Dummy-столбцы:", list(room_dummies.columns))
print("В каждой строке ровно одна единица:", (room_dummies.sum(axis=1) == 1).all())

Dummy-столбцы: ['room_Entire_home_apt', 'room_Hotel_room', 'room_Private_room', 'room_Shared_room']
В каждой строке ровно одна единица: True


**Почему One-Hot лучше, чем 0, 1, 2, ...** Типы жилья — номинальная категория: «Entire home/apt», «Hotel room», «Private room» и «Shared room» нельзя упорядочить по числовой шкале. Если присвоить им числа 0, 1, 2, 3, модель будет считать, что, например, `Shared room` (3) «втрое больше» `Hotel room` (1) и что между типами есть равные шаги — это ложная информация. При One-Hot каждый тип — отдельный независимый признак 0/1, и модель может выучить собственный эффект для каждого типа жилья.

Заметим, что столбец `room_Entire_home_apt` совпадает с созданным в 3.2 `is_entire_home` — это тот же признак, полученный другим способом.

## 7. Масштабирование

Берём копию четырёх числовых признаков. Масштабирование применяем к копиям, чтобы `df_work` остался в исходных единицах.

In [21]:
scale_columns = ["price", "minimum_nights", "number_of_reviews", "availability_365"]
features = df_work[scale_columns].copy()

features.describe().round(2)

,price,minimum_nights,number_of_reviews,availability_365
count,21796.00,21796.00,21796.00,21796.00
mean,278.33,24.19,39.08,246.59
std,528.55,18.26,89.95,104.98
min,5.00,1.00,0.00,1.00
25%,98.00,30.00,0.00,165.00
50%,176.00,30.00,6.00,274.00
75%,304.00,30.00,40.00,337.00
max,30973.00,365.00,4502.00,365.00


### 7.1. `StandardScaler`

Преобразование `z = (x − среднее) / стандартное отклонение` для каждого признака.

In [22]:
standard_scaler = StandardScaler()
features_standard = pd.DataFrame(
    standard_scaler.fit_transform(features),
    columns=scale_columns,
    index=features.index,
)

features_standard.describe().round(2)

,price,minimum_nights,number_of_reviews,availability_365
count,21796.00,21796.00,21796.00,21796.00
mean,-0.00,-0.00,0.00,0.00
std,1.00,1.00,1.00,1.00
min,-0.52,-1.27,-0.43,-2.34
25%,-0.34,0.32,-0.43,-0.78
50%,-0.19,0.32,-0.37,0.26
75%,0.05,0.32,0.01,0.86
max,58.07,18.66,49.62,1.13


In [23]:
# Проверяем, сохранился ли порядок объектов: ранги значений до и после должны совпасть
order_preserved = (features.rank() == features_standard.rank()).all()
print("Порядок объектов сохранился:")
print(order_preserved)

Порядок объектов сохранился:
price                True
minimum_nights       True
number_of_reviews    True
availability_365     True
dtype: bool


**Что произошло после `StandardScaler`:**
- **Средние** всех четырёх признаков стали равны 0 (в `describe()` видно 0.00, отличия от нуля — только ошибки округления).
- **Стандартные отклонения** стали равны 1 — все признаки теперь измеряются в «стандартных отклонениях от среднего», а не в долларах, ночах, отзывах и днях.
- **Порядок объектов сохранился** — ранги совпали для всех признаков: преобразование линейное и возрастающее, поэтому самое дорогое объявление осталось самым дорогим.

При этом форма распределения не изменилась: у `price` и `number_of_reviews` по-прежнему длинный хвост — максимальные значения после стандартизации очень велики (десятки стандартных отклонений), потому что `StandardScaler` не устраняет выбросы.

### 7.2. `MinMaxScaler`

Преобразование `(x − min) / (max − min)` приводит каждый признак к диапазону [0, 1].

In [24]:
minmax_scaler = MinMaxScaler()
features_minmax = pd.DataFrame(
    minmax_scaler.fit_transform(features),
    columns=scale_columns,
    index=features.index,
)

features_minmax.agg(["min", "max", "median"]).round(4)

,price,minimum_nights,number_of_reviews,availability_365
min,0.0000,0.0000,0.0000,0.00
max,1.0000,1.0000,1.0000,1.00
median,0.0055,0.0797,0.0013,0.75


**Проверка.** Минимум каждого признака после преобразования равен 0, максимум — 1. Но обратите внимание на медианы: у `price` она около 0.005, у `number_of_reviews` — около 0.001, у `minimum_nights` — около 0.08. Из-за единичных экстремальных значений (цена ~31 000 $ за ночь, 4 502 отзыва, минимальный срок 365 ночей при типичных 30) почти все объекты оказались прижаты к нулю. У `availability_365` без выбросов значения распределены по всему диапазону.

### 7.3. Сравнение `StandardScaler` и `MinMaxScaler`

- **`StandardScaler`** центрирует признак (среднее → 0) и делает стандартное отклонение равным 1. Диапазон значений не ограничен: выбросы остаются большими по модулю числами. Подходит, когда важно сравнивать отклонения от среднего и для моделей, которые предполагают центрированные данные.
- **`MinMaxScaler`** сдвигает и растягивает признак так, чтобы минимум стал 0, а максимум — 1. Все значения попадают в [0, 1], но метод очень чувствителен к выбросам: один экстремальный объект «сжимает» все остальные к краю диапазона, как это произошло с `price`.

Оба метода линейные и сохраняют порядок объектов и форму распределения; различаются только тем, относительно чего масштабируют — среднего и разброса или минимума и максимума.

## 8. Финальная таблица признаков

Собираем таблицу для будущей модели:
- оставляем `id` как идентификатор;
- исходные числовые признаки (`price`, `minimum_nights`, `number_of_reviews`, `reviews_per_month`, `availability_365`) и созданные `name_length`, `mentions_view`;
- `room_type` заменяем dummy-столбцами (задание 6); `is_entire_home` не дублируем — он совпадает с `room_Entire_home_apt`;
- `neighbourhood_group` кодируем через One-Hot, как обосновано в 3.3; текстовый `borough_code` в финальную таблицу не идёт;
- `stay_type` и `review_activity` — в отличие от районов, у них **есть естественный порядок** (короткий < средний < долгий срок; нет отзывов < редкие < активные), поэтому для них подходит порядковое кодирование числами 0, 1, 2 через `map`;
- `price_per_week` не включаем: это `price × 7`, он не несёт новой информации;
- `reviews_per_month` заполняем нулём — проверяем, что пропуск бывает только у объявлений без отзывов.

Числовые признаки оставлены в исходных единицах: масштабирование обычно выполняют уже на этапе обучения модели, после разделения на train/test (в этой работе его не требуется).

In [25]:
print(
    "Пропуск reviews_per_month только у объявлений без отзывов:",
    (df_work["reviews_per_month"].isna() == (df_work["number_of_reviews"] == 0)).all(),
)

Пропуск reviews_per_month только у объявлений без отзывов: True


In [26]:
borough_dummies = pd.get_dummies(df_work["neighbourhood_group"], prefix="borough", dtype=int)
borough_dummies.columns = borough_dummies.columns.str.replace(" ", "_")

stay_type_order = {"short": 0, "medium": 1, "long": 2}
review_activity_order = {"no_reviews": 0, "rare": 1, "active": 2}

final_features = pd.concat(
    [
        df_work[["id", "price", "minimum_nights", "number_of_reviews", "availability_365",
                 "name_length", "mentions_view"]],
        df_work["reviews_per_month"].fillna(0),
        df_work["stay_type"].map(stay_type_order).rename("stay_type_code"),
        df_work["review_activity"].map(review_activity_order).rename("review_activity_code"),
        room_dummies,
        borough_dummies,
    ],
    axis=1,
).reset_index(drop=True)

print("Размер финальной таблицы:", final_features.shape)
final_features.head()

Размер финальной таблицы: (21796, 19)


,id,price,minimum_nights,number_of_reviews,availability_365,name_length,mentions_view,reviews_per_month,stay_type_code,review_activity_code,room_Entire_home_apt,room_Hotel_room,room_Private_room,room_Shared_room,borough_Bronx,borough_Brooklyn,borough_Manhattan,borough_Queens,borough_Staten_Island
0,1279666081527148590,468.0,1.0,3,338,24,0,0.57,0,1,0,0,1,0,0,0,1,0,0
1,1543683559947083167,188.0,1.0,23,265,50,0,3.37,0,2,0,0,1,0,0,0,1,0,0
2,1559522395930572140,431.0,1.0,3,273,33,0,0.54,0,1,0,0,1,0,0,0,1,0,0
3,1665948589959289542,257.0,1.0,1,360,46,0,1.00,0,1,0,0,1,0,0,0,1,0,0
4,1394234603510215650,170.0,1.0,168,365,43,0,12.23,0,2,0,0,1,0,0,0,1,0,0


In [27]:
print("Типы данных финальной таблицы:")
print(final_features.dtypes)
print()
print("Пропусков в финальной таблице:", final_features.isna().sum().sum())
print("Текстовых столбцов room_type / neighbourhood_group нет:",
      "room_type" not in final_features.columns and "neighbourhood_group" not in final_features.columns)
print("id уникальны:", final_features["id"].is_unique)

Типы данных финальной таблицы:
id                         int64
price                    float64
minimum_nights           float64
number_of_reviews          int64
availability_365           int64
name_length                int64
mentions_view              int64
reviews_per_month        float64
stay_type_code             int64
review_activity_code       int64
room_Entire_home_apt       int64
room_Hotel_room            int64
room_Private_room          int64
room_Shared_room           int64
borough_Bronx              int64
borough_Brooklyn           int64
borough_Manhattan          int64
borough_Queens             int64
borough_Staten_Island      int64
dtype: object

Пропусков в финальной таблице: 0
Текстовых столбцов room_type / neighbourhood_group нет: True
id уникальны: True


**Вывод.** Итоговая таблица содержит 21 796 объявлений и только числовые признаки без пропусков: `id` как идентификатор, исходные числовые характеристики, созданные признаки и закодированные категории.

1. **Векторными** были `price_per_week = price * 7`, `name_length` через `.str.len()`, условное заполнение `stay_type` через `loc` по маскам, а также One-Hot Encoding и масштабирование — всё это операции над целыми столбцами сразу.
2. **`map`** понадобился для замены значений по словарю: коды районов `borough_code`, бинарный `is_entire_home` и порядковое кодирование `stay_type` и `review_activity` в финальной таблице.
3. **`apply` действительно был нужен** только для `review_activity`: правило зависит сразу от двух столбцов и содержит несколько последовательных условий с обработкой пропуска, поэтому его удобнее записать отдельной функцией и применить к каждой строке через `axis=1`.
4. **`lambda` не стоит использовать для каждой операции**: она работает построчно и медленнее векторных методов, а сложная логика в одной строке плохо читается и не переиспользуется — `lambda` уместна только для коротких разовых правил вроде `mentions_view`.
5. **Масштабируют числовые признаки**, потому что они измеряются в разных единицах и диапазонах (доллары до ~31 000, дни до 365, отзывы до ~4 500): без масштабирования признаки с большими числами доминировали бы в моделях, основанных на расстояниях или градиентном спуске, только из-за своих единиц измерения.